In [ ]:
import sounddevice as sd
import numpy as np
from scipy.io.wavfile import write
from io import BytesIO
import librosa
import torch
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings

# Load model and processor
model = Qwen2AudioForConditionalGeneration.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct", device_map="auto")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")# Suppress all warnings
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct")

In [ ]:
def record_audio(duration=5, fs=16000):
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype='float32')
    sd.wait()
    print("Recording finished.")
    return audio.flatten(), fs

In [ ]:
# Initialize conversation
conversation = [
    {'role': 'system', 'content': 'You are a helpful assistant. The user will speak with you, i want first to translate into text what you hear, then answer the question he addresses'},
]

while True:
    # Record from microphone
    audio_data, _ = record_audio(duration=5, fs=processor.feature_extractor.sampling_rate)

    conversation = [msg for msg in conversation if not (msg["role"] == "user" and isinstance(msg["content"], list) and msg["content"][0]["type"] == "audio")]

    # Add current audio message
    conversation.append({
        "role": "user",
        "content": [{"type": "audio", "audio_url": None}]
    })

    # Format text input for the model
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)

    # Prepare inputs
    inputs = processor(text=text, audios=[audio_data], return_tensors="pt", padding=True)
    inputs.input_ids = inputs.input_ids.to("cuda")

    # Generate response
    generate_ids = model.generate(**inputs, max_length=512)
    generate_ids = generate_ids[:, inputs.input_ids.size(1):]
    response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

    print(f"\n🧠 Assistant: {response}")

    # Append response to conversation
    conversation.append({"role": "assistant", "content": response})

    print("\n--- Speak again or Ctrl+C to exit ---")